In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import re
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "flexi_mod").exists():
            return candidate
    raise RuntimeError("Could not find FLEXIMOD repository root.")


def get_example_from_run_case(repo_root: Path) -> str:
    """Lese den 'example' Wert aus run_case.py aus."""
    run_case_path = repo_root / "src" / "flexi_mod" / "simulation" / "run_case.py"
    with open(run_case_path) as f:
        content = f.read()
    match = re.search(r'example\s*=\s*["\']([^"\']+)["\']', content)
    if match:
        return match.group(1)
    raise ValueError("Could not find 'example' variable in run_case.py")


REPO_ROOT = find_repo_root()
SELECTED_CASE = get_example_from_run_case(REPO_ROOT)
INPUT_CASE_DIR = REPO_ROOT / "data" / "input" / SELECTED_CASE
OUTPUT_ROOT = REPO_ROOT / "data" / "output"
NOTEBOOK_PATH = REPO_ROOT / "notebooks" / "fleximod_report_analysis.ipynb"
REPORT_HTML_PATH = OUTPUT_ROOT / "reports" / "fleximod_report_analysis.html"
REPORT_SCENARIO = f"{SELECTED_CASE}_hybrid_etes_gas"
COMPARISON_SCENARIOS = [f"{SELECTED_CASE}_hybrid_etes_gas"]
SCENARIO_LABELS = {
    f"{SELECTED_CASE}_hybrid_etes_gas": "DA + IDC + aFRR energy + capacity",
}
EXPORT_HTML = True

COLORS = {
    "dark": "#0E3E3E",
    "teal": "#229797",
    "green": "#00DA9D",
    "orange": "#F28E2B",
    "red": "#F33401",
    "blue": "#4C78A8",
    "purple": "#8B6FB3",
    "cyan": "#63B7CC",
    "brown": "#9D7F64",
    "grid": "#E8ECEF",
}
pio.templates["fleximod_report"] = go.layout.Template(
    layout=go.Layout(
        font=dict(family="Arial, sans-serif", size=13, color="#263238"),
        title=dict(font=dict(size=22, color=COLORS["dark"]), x=0.02),
        paper_bgcolor="white",
        plot_bgcolor="white",
        colorway=[
            COLORS[k] for k in ["dark", "green", "orange", "red", "blue", "purple", "cyan", "brown"]
        ],
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0),
        margin=dict(l=70, r=45, t=90, b=70),
        xaxis=dict(showgrid=True, gridcolor=COLORS["grid"], zerolinecolor="#B0BEC5"),
        yaxis=dict(showgrid=True, gridcolor=COLORS["grid"], zerolinecolor="#B0BEC5"),
    )
)
pio.templates.default = "fleximod_report"
# Ensure Plotly figures are represented as HTML during nbconvert export.
pio.renderers.default = "notebook"


def parse_datetime(frame: pd.DataFrame, column: str = "datetime") -> pd.DataFrame:
    if frame.empty or column not in frame.columns:
        return frame.copy()
    parsed = frame.copy()
    parsed[column] = pd.to_datetime(parsed[column], errors="coerce", utc=True).dt.tz_convert(None)
    return parsed.dropna(subset=[column]).sort_values(column)


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def load_scenario(folder: Path) -> dict[str, pd.DataFrame]:
    tables = {
        "summary": read_csv(folder / "summary_indicators.csv"),
        "dispatch": parse_datetime(read_csv(folder / "dispatch_results.csv")),
        "market": parse_datetime(read_csv(folder / "market_ledger.csv")),
        "storage": parse_datetime(read_csv(folder / "storage_cost_ledger.csv")),
        "capacity": read_csv(folder / "afrr_capacity_block_summary.csv"),
    }
    if not tables["capacity"].empty:
        for col in ["block_start", "block_end"]:
            if col in tables["capacity"].columns:
                tables["capacity"][col] = pd.to_datetime(
                    tables["capacity"][col], errors="coerce", utc=True
                ).dt.tz_convert(None)
    return tables


def value(frame: pd.DataFrame, column: str, default: float = 0.0) -> float:
    if frame.empty or column not in frame.columns:
        return default
    data = pd.to_numeric(frame[column], errors="coerce").dropna()
    return float(data.iloc[0]) if not data.empty else default


def series(frame: pd.DataFrame, column: str, default: float = 0.0) -> pd.Series:
    if frame.empty:
        return pd.Series(dtype=float)
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce").fillna(default)


def month_sum(frame: pd.DataFrame, column: str) -> pd.Series:
    if frame.empty or "datetime" not in frame.columns:
        return pd.Series(dtype=float)
    data = pd.DataFrame({"datetime": frame["datetime"], "value": series(frame, column)})
    return data.dropna(subset=["datetime"]).set_index("datetime")["value"].resample("ME").sum()


def month_mean(frame: pd.DataFrame, column: str) -> pd.Series:
    if frame.empty or "datetime" not in frame.columns:
        return pd.Series(dtype=float)
    data = pd.DataFrame({"datetime": frame["datetime"], "value": series(frame, column)})
    return data.dropna(subset=["datetime"]).set_index("datetime")["value"].resample("ME").mean()


def kEUR(x: float) -> float:
    return x / 1_000.0


def style_table(frame: pd.DataFrame, caption: str, precision: int = 2):
    display(frame.style.format(precision=precision, thousands=",").set_caption(caption))


def finish(fig: go.Figure, title: str, height: int = 560) -> go.Figure:
    fig.update_layout(title=title, height=height, hovermode="x unified")
    return fig


def cards(items: list[tuple[str, str, str]]) -> HTML:
    body = []
    for title, number, subtitle in items:
        body.append(f"""
        <div style='background:white;border-left:5px solid {COLORS["green"]};box-shadow:0 4px 16px rgba(14,62,62,0.12);border-radius:12px;padding:16px 18px;min-width:210px;'>
          <div style='font-size:13px;color:#546E7A;font-weight:600;'>{title}</div>
          <div style='font-size:26px;color:{COLORS["dark"]};font-weight:700;margin-top:6px;'>{number}</div>
          <div style='font-size:12px;color:#78909C;margin-top:4px;'>{subtitle}</div>
        </div>""")
    return HTML(
        "<div style='display:flex;flex-wrap:wrap;gap:14px;margin:10px 0 24px 0;'>"
        + "".join(body)
        + "</div>"
    )

## 1. Load Data And Build Scenario Table


In [ ]:
forecasts = parse_datetime(read_csv(INPUT_CASE_DIR / "forecasts_df.csv"))
plants = read_csv(INPUT_CASE_DIR / "plants.csv")
additional_charges = read_csv(INPUT_CASE_DIR / "additional_charges.csv")

scenario_outputs = {}
for scenario in COMPARISON_SCENARIOS:
    folder = OUTPUT_ROOT / scenario
    if folder.exists():
        scenario_outputs[scenario] = load_scenario(folder)
    else:
        warnings.warn(f"Missing output folder: {folder}", stacklevel=1)

if REPORT_SCENARIO not in scenario_outputs:
    raise FileNotFoundError(f"Report scenario output folder not found: {REPORT_SCENARIO}")

report = scenario_outputs[REPORT_SCENARIO]
dispatch = report["dispatch"]
market = report["market"]
summary = report["summary"]
capacity = report["capacity"]


def boiler_efficiency(default: float = 0.9) -> float:
    if plants.empty or "technology" not in plants.columns or "efficiency" not in plants.columns:
        return default
    boiler = plants[plants["technology"].astype(str).str.lower().str.contains("boiler")]
    eff = pd.to_numeric(boiler.get("efficiency", pd.Series(dtype=float)), errors="coerce").dropna()
    return float(eff.iloc[0]) if not eff.empty else default


def gas_only_cost(frame: pd.DataFrame) -> float:
    if frame.empty:
        return 0.0
    return float(
        (
            series(frame, "heat_demand_MWh")
            / boiler_efficiency()
            * series(frame, "gas_price_EUR_per_MWh")
        ).sum()
    )


def metrics(name: str, tables: dict[str, pd.DataFrame]) -> dict[str, float | str]:
    s, d = tables["summary"], tables["dispatch"]
    gas = value(s, "total_gas_cost_EUR")
    da = float(series(d, "DA_electricity_cost_EUR").sum())
    id_buy = float(series(d, "IDC_buy_cost_EUR").sum())
    id_sell = value(s, "IDC_sell_revenue_EUR", float(series(d, "IDC_sell_revenue_EUR").sum()))
    afrr_energy = value(s, "afrr_energy_cost_EUR", float(series(d, "afrr_energy_cost_EUR").sum()))
    charges = value(s, "total_additional_electricity_charges_cost_EUR")
    cap_rev = value(s, "total_afrr_capacity_revenue_EUR")
    net = value(s, "net_operating_cost_EUR", value(s, "total_net_operating_cost_EUR"))
    if net == 0.0:
        net = gas + da + id_buy + afrr_energy + charges - id_sell - cap_rev
    benchmark = gas_only_cost(d)
    return {
        "scenario_key": name,
        "scenario": SCENARIO_LABELS.get(name, name),
        "heat_demand_MWh": value(s, "total_heat_demand_MWh"),
        "gas_heat_MWh": value(s, "total_gas_heat_MWh"),
        "electric_heat_MWh": value(s, "total_storage_discharge_heat_MWh"),
        "etes_charge_MWh_el": value(s, "total_etes_charging_electricity_MWh"),
        "DA_procured_MWh_el": value(s, "total_DA_electricity_MWh"),
        "IDC_buy_MWh_el": value(s, "total_IDC_buy_MWh"),
        "IDC_sell_MWh_el": value(s, "total_IDC_sell_MWh"),
        "afrr_energy_activated_MWh_el": value(s, "total_afrr_energy_activated_MWh"),
        "afrr_capacity_reserved_MW_h": value(s, "total_afrr_capacity_reserved_MW_h"),
        "gas_only_benchmark_cost_EUR": benchmark,
        "gas_cost_EUR": gas,
        "DA_market_cost_EUR": da,
        "IDC_buy_cost_EUR": id_buy,
        "IDC_sell_revenue_EUR": id_sell,
        "afrr_energy_cost_EUR": afrr_energy,
        "afrr_energy_market_reward_EUR": value(s, "total_afrr_energy_pay_as_cleared_reward_EUR"),
        "afrr_energy_net_value_after_charges_EUR": value(
            s, "total_afrr_energy_net_value_after_charges_EUR"
        ),
        "afrr_capacity_revenue_EUR": cap_rev,
        "additional_electricity_charges_EUR": charges,
        "net_operating_cost_EUR": net,
        "cost_savings_vs_gas_only_EUR": benchmark - net,
    }


analysis_df = pd.DataFrame([metrics(name, tables) for name, tables in scenario_outputs.items()])
analysis_df["scenario"] = pd.Categorical(
    analysis_df["scenario"],
    categories=[
        SCENARIO_LABELS.get(name, name) for name in COMPARISON_SCENARIOS if name in scenario_outputs
    ],
    ordered=True,
)
analysis_df = analysis_df.sort_values("scenario")
report_row = analysis_df[analysis_df["scenario_key"] == REPORT_SCENARIO].iloc[0]

loaded = pd.DataFrame(
    [
        {
            "scenario": SCENARIO_LABELS.get(name, name),
            "dispatch rows": len(tables["dispatch"]),
            "market rows": len(tables["market"]),
            "capacity blocks": len(tables["capacity"]),
        }
        for name, tables in scenario_outputs.items()
    ]
)
style_table(loaded, "Loaded scenario outputs", precision=0)

# Executive Summary

Core operational and economic indicators for the selected full-market report scenario


In [ ]:
display(
    cards(
        [
            ("Heat demand", f"{report_row['heat_demand_MWh']:,.0f} MWh", "simulation period"),
            ("Gas heat", f"{report_row['gas_heat_MWh']:,.0f} MWh", "final dispatch"),
            ("ETES heat", f"{report_row['electric_heat_MWh']:,.0f} MWh", "storage discharge"),
            (
                "ETES charging",
                f"{report_row['etes_charge_MWh_el']:,.0f} MWh_el",
                "electricity consumption",
            ),
            (
                "Net operating cost",
                f"{kEUR(report_row['net_operating_cost_EUR']):,.0f} kEUR",
                "after revenues",
            ),
            (
                "Savings vs gas-only",
                f"{kEUR(report_row['cost_savings_vs_gas_only_EUR']):,.0f} kEUR",
                "benchmark comparison",
            ),
        ]
    )
)

baseline_cost = report_row["gas_only_benchmark_cost_EUR"]
net_cost = report_row["net_operating_cost_EUR"]
savings = report_row["cost_savings_vs_gas_only_EUR"]
savings_pct = 100.0 * savings / baseline_cost if baseline_cost > 0 else 0.0
elec_rate = (
    100.0 * report_row["electric_heat_MWh"] / report_row["heat_demand_MWh"]
    if report_row["heat_demand_MWh"] > 0
    else 0.0
)

kpi_cards = f"""
<div style='display:grid;grid-template-columns:repeat(2,1fr);gap:16px;margin:16px 0 24px 0;'>
  <div style='background:{COLORS["green"]};border-left:5px solid {COLORS["green"]};border-radius:12px;padding:20px;box-shadow:0 4px 16px rgba(14,62,62,0.12);'>
    <div style='font-size:13px;color:white;font-weight:600;opacity:0.9;'>Total Savings vs Gas Baseline</div>
    <div style='font-size:32px;color:white;font-weight:700;margin-top:8px;'>{kEUR(savings):,.0f}</div>
    <div style='font-size:12px;color:white;opacity:0.85;margin-top:4px;'>kEUR ({savings_pct:.1f}% reduction)</div>
  </div>
  <div style='background:#F0F4F7;border-left:5px solid {COLORS["teal"]};border-radius:12px;padding:20px;box-shadow:0 4px 16px rgba(14,62,62,0.12);'>
    <div style='font-size:13px;color:#546E7A;font-weight:600;'>Baseline Gas Cost</div>
    <div style='font-size:32px;color:{COLORS["dark"]};font-weight:700;margin-top:8px;'>{kEUR(baseline_cost):,.0f}</div>
    <div style='font-size:12px;color:#78909C;margin-top:4px;'>kEUR (gas-only scenario)</div>
  </div>
  <div style='background:#F0F4F7;border-left:5px solid {COLORS["teal"]};border-radius:12px;padding:20px;box-shadow:0 4px 16px rgba(14,62,62,0.12);'>
    <div style='font-size:13px;color:#546E7A;font-weight:600;'>Energy Cost with FLEXIMOD</div>
    <div style='font-size:32px;color:{COLORS["dark"]};font-weight:700;margin-top:8px;'>{kEUR(net_cost):,.0f}</div>
    <div style='font-size:12px;color:#78909C;margin-top:4px;'>kEUR (including all markets)</div>
  </div>
  <div style='background:#F0F4F7;border-left:5px solid {COLORS["teal"]};border-radius:12px;padding:20px;box-shadow:0 4px 16px rgba(14,62,62,0.12);'>
    <div style='font-size:13px;color:#546E7A;font-weight:600;'>Electrification Rate</div>
    <div style='font-size:32px;color:{COLORS["dark"]};font-weight:700;margin-top:8px;'>{elec_rate:.1f}</div>
    <div style='font-size:12px;color:#78909C;margin-top:4px;'>% ({report_row["electric_heat_MWh"]:,.0f} MWh)</div>
  </div>
</div>
"""

display(HTML(kpi_cards))

view = analysis_df[
    [
        "scenario",
        "gas_cost_EUR",
        "DA_market_cost_EUR",
        "IDC_buy_cost_EUR",
        "afrr_energy_cost_EUR",
        "additional_electricity_charges_EUR",
        "IDC_sell_revenue_EUR",
        "afrr_capacity_revenue_EUR",
        "net_operating_cost_EUR",
        "cost_savings_vs_gas_only_EUR",
    ]
].copy()
for col in view.columns:
    if col != "scenario":
        view[col] = view[col].map(kEUR)
style_table(view, "Scenario economics [kEUR]", precision=1)

# System Setup

Plant configuration: the thermal demand profile is classified as **constant** or **time-variant** from its coefficient of variation, alongside the installed e-heater power, thermal storage capacity, and the gas-boiler backup retained for hybrid-mode operation.

In [ ]:
def _to_float(x, default=np.nan):
    v = pd.to_numeric(pd.Series([x]), errors="coerce").iloc[0]
    return float(v) if pd.notna(v) else default


def plant_sizing() -> dict[str, float]:
    sizing = {
        "eheater_power_MW": np.nan,
        "discharge_power_MW": np.nan,
        "storage_capacity_MWh": np.nan,
        "gas_boiler_power_MW": np.nan,
        "gas_boiler_efficiency": np.nan,
        "charge_efficiency": np.nan,
        "discharge_efficiency": np.nan,
    }
    if plants.empty or "technology" not in plants.columns:
        return sizing
    tech = plants["technology"].astype(str).str.lower()
    storage_rows = plants[tech.str.contains("storage")]
    boiler_rows = plants[tech.str.contains("boiler")]
    if not storage_rows.empty:
        row = storage_rows.iloc[0]
        charge = _to_float(row.get("max_power_charge"))
        sizing["eheater_power_MW"] = (
            charge if np.isfinite(charge) else _to_float(row.get("max_power"))
        )
        sizing["discharge_power_MW"] = _to_float(row.get("max_power_discharge"))
        sizing["storage_capacity_MWh"] = _to_float(row.get("max_capacity"))
        sizing["charge_efficiency"] = _to_float(row.get("efficiency_charge"))
        sizing["discharge_efficiency"] = _to_float(row.get("efficiency_discharge"))
    if not boiler_rows.empty:
        row_b = boiler_rows.iloc[0]
        sizing["gas_boiler_power_MW"] = _to_float(row_b.get("max_power"))
        sizing["gas_boiler_efficiency"] = _to_float(row_b.get("efficiency"))
    return sizing


SIZING = plant_sizing()

heat_series = series(dispatch, "heat_demand_MWh")
heat_series = heat_series[heat_series.notna()]
mean_heat = float(heat_series.mean()) if not heat_series.empty else 0.0
cv_heat = float(heat_series.std() / mean_heat) if mean_heat else 0.0
profile_type = "Constant" if cv_heat < 0.05 else "Time-variant"
peak_heat = float(heat_series.max()) if not heat_series.empty else 0.0

eheater_mw = SIZING["eheater_power_MW"]
discharge_mw = SIZING["discharge_power_MW"]
storage_mwh = SIZING["storage_capacity_MWh"]
boiler_mw = SIZING["gas_boiler_power_MW"]
boiler_eff = SIZING["gas_boiler_efficiency"]
eta_charge = SIZING["charge_efficiency"]
eta_discharge = SIZING["discharge_efficiency"]
roundtrip_eff = (
    eta_charge * eta_discharge if np.isfinite(eta_charge) and np.isfinite(eta_discharge) else np.nan
)
storage_hours = (
    storage_mwh / eheater_mw
    if np.isfinite(eheater_mw) and eheater_mw > 0 and np.isfinite(storage_mwh)
    else np.nan
)

display(
    cards(
        [
            ("Thermal profile", profile_type, f"CV {cv_heat:.0%} - peak {peak_heat:,.1f} MWh/step"),
            ("E-heater charge power", f"{eheater_mw:,.1f} MW_el", "storage charging capacity"),
            (
                "E-heater discharge power",
                f"{discharge_mw:,.1f} MW_th",
                "storage discharging capacity",
            ),
            (
                "Storage capacity",
                f"{storage_mwh:,.1f} MWh_th",
                f"~{storage_hours:,.1f} h at full e-heater power",
            ),
            ("Charge efficiency (η_c)", f"{eta_charge:.0%}", "electricity → stored heat"),
            ("Discharge efficiency (η_d)", f"{eta_discharge:.0%}", "stored heat → process heat"),
            ("Round-trip efficiency", f"{roundtrip_eff:.0%}", "η_charge × η_discharge"),
            ("Gas-boiler efficiency", f"{boiler_eff:.0%}", "gas → thermal heat"),
            ("Gas-boiler backup power", f"{boiler_mw:,.1f} MW_th", "hybrid-mode / fallback supply"),
        ]
    )
)

setup_table = pd.DataFrame(
    [
        {
            "parameter": "Thermal demand profile",
            "value": profile_type,
            "unit": f"CV = {cv_heat:.1%}",
        },
        {"parameter": "Mean heat demand", "value": round(mean_heat, 3), "unit": "MWh_th / step"},
        {"parameter": "Peak heat demand", "value": round(peak_heat, 3), "unit": "MWh_th / step"},
        {"parameter": "E-heater charge power", "value": round(eheater_mw, 3), "unit": "MW_el"},
        {"parameter": "E-heater discharge power", "value": round(discharge_mw, 3), "unit": "MW_th"},
        {"parameter": "Storage capacity", "value": round(storage_mwh, 3), "unit": "MWh_th"},
        {
            "parameter": "Storage duration at full power",
            "value": round(storage_hours, 2),
            "unit": "h",
        },
        {"parameter": "Charge efficiency (η_c)", "value": round(eta_charge, 3), "unit": "-"},
        {"parameter": "Discharge efficiency (η_d)", "value": round(eta_discharge, 3), "unit": "-"},
        {"parameter": "Round-trip efficiency", "value": round(roundtrip_eff, 3), "unit": "-"},
        {"parameter": "Gas-boiler efficiency", "value": round(boiler_eff, 3), "unit": "-"},
        {"parameter": "Gas-boiler backup power", "value": round(boiler_mw, 3), "unit": "MW_th"},
    ]
)
style_table(setup_table, "System setup summary", precision=3)

# 2. Input Analysis

Thermal demand, price signals, additional electricity charges, and the gas-based electricity benchmark used by the strategy.


In [ ]:
def demand_frame() -> pd.DataFrame:
    demand_col = None
    if not plants.empty and "demand" in plants.columns:
        candidates = [
            x for x in plants["demand"].dropna().astype(str).unique() if x in forecasts.columns
        ]
        demand_col = candidates[0] if candidates else None
    if demand_col and "datetime" in forecasts.columns:
        frame = (
            forecasts[["datetime", demand_col]]
            .rename(columns={demand_col: "thermal_demand"})
            .copy()
        )
    else:
        frame = (
            dispatch[["datetime", "heat_demand_MWh"]]
            .rename(columns={"heat_demand_MWh": "thermal_demand"})
            .copy()
        )
    frame["thermal_demand"] = pd.to_numeric(frame["thermal_demand"], errors="coerce").fillna(0.0)
    return frame


demand = demand_frame()
# Infer timestep duration in hours from the datetime index so the MW conversion is resolution-agnostic
_dt = demand["datetime"].dropna().sort_values()
_step_h = float((_dt.iloc[1] - _dt.iloc[0]).total_seconds() / 3600) if len(_dt) > 1 else 0.25
demand["thermal_demand_MW"] = demand["thermal_demand"] / _step_h

fig = go.Figure(
    go.Scatter(
        x=demand["datetime"],
        y=demand["thermal_demand_MW"],
        mode="lines",
        name="Thermal demand",
        line=dict(color=COLORS["red"], width=1.3),
    )
)
fig.update_yaxes(title_text="Thermal Demand (MW)")
fig.update_xaxes(title_text="Time")
finish(fig, "Thermal demand time-series", 500).show()

load_curve_mw = demand["thermal_demand_MW"].sort_values(ascending=False).reset_index(drop=True)
hours_of_year = np.arange(1, len(load_curve_mw) + 1) * _step_h
fig = go.Figure(
    go.Scatter(
        x=hours_of_year,
        y=load_curve_mw,
        mode="lines",
        fill="tozeroy",
        name="Thermal Demand",
        line=dict(color=COLORS["red"], width=2),
        fillcolor="rgba(243,52,1,0.20)",
    )
)
fig.update_yaxes(title_text="Thermal Demand (MW)")
fig.update_xaxes(title_text="Hours of Year")
finish(fig, "Thermal load duration curve", 500).show()

h = demand.copy()
h["date"] = h["datetime"].dt.date
h["hour"] = h["datetime"].dt.hour + h["datetime"].dt.minute / 60
heatmap = h.pivot_table(index="date", columns="hour", values="thermal_demand_MW", aggfunc="mean")
fig = go.Figure(
    go.Heatmap(
        z=heatmap.values,
        x=[f"{x:g}" for x in heatmap.columns],
        y=[str(x) for x in heatmap.index],
        colorscale=[
            [0, "#FFFFFF"],
            [0.25, "#FFC8B8"],
            [0.5, "#FF9580"],
            [0.75, "#FF6247"],
            [1, COLORS["red"]],
        ],
        colorbar=dict(title="MW"),
    )
)
fig.update_xaxes(title_text="Hour of day")
fig.update_yaxes(title_text="Date", nticks=14)
finish(fig, "Thermal demand heatmap", 650).show()

hourly = h.groupby(h["datetime"].dt.hour)["thermal_demand_MW"].mean()
weekday = (
    h.groupby(h["datetime"].dt.day_name())["thermal_demand_MW"]
    .mean()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Average by hour", "Average by weekday"))
fig.add_trace(
    go.Scatter(
        x=hourly.index,
        y=hourly.values,
        mode="lines+markers",
        name="Hour",
        line=dict(color=COLORS["dark"], width=3),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=weekday.index, y=weekday.values, name="Weekday", marker_color=COLORS["green"]),
    row=1,
    col=2,
)
fig.update_yaxes(title_text="Average demand [MW]", row=1, col=1)
fig.update_yaxes(title_text="Average demand [MW]", row=1, col=2)
finish(fig, "Thermal demand temporal patterns", 520).show()

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
h_dow = demand.copy()
h_dow["day_of_week"] = h_dow["datetime"].dt.day_name()
h_dow["hour"] = h_dow["datetime"].dt.hour

heatmap_dow = (
    h_dow.groupby(["day_of_week", "hour"])["thermal_demand_MW"]
    .mean()
    .unstack(level="hour")
    .reindex(dow_order)
)

fig = go.Figure(
    go.Heatmap(
        z=heatmap_dow.values,
        x=heatmap_dow.columns.tolist(),
        y=heatmap_dow.index.tolist(),
        colorscale=[
            [0, "#FFFFFF"],
            [0.25, "#FFC8B8"],
            [0.5, "#FF9580"],
            [0.75, "#FF6247"],
            [1, COLORS["red"]],
        ],
        colorbar=dict(title="Thermal Demand (MW)"),
        hovertemplate="<b>%{y}</b><br>Hour: %{x}:00<br>Avg demand: %{z:.2f} MW<extra></extra>",
    )
)
fig.update_xaxes(title_text="Hour of Day", tickmode="linear", tick0=0, dtick=1)
fig.update_yaxes(title_text="Day of Week")
finish(fig, "Average Demand Heatmap: Hour vs Day of Week", 500).show()

In [ ]:
fig = go.Figure()
for col, label, color in [
    ("day_ahead_price_EUR_per_MWh", "DA market price", COLORS["blue"]),
    ("day_ahead_delivered_price_EUR_per_MWh", "DA delivered price", COLORS["orange"]),
    ("IDC_delivered_price_EUR_per_MWh", "IDC delivered price", COLORS["green"]),
    ("afrr_energy_delivered_price_EUR_per_MWh", "aFRR delivered price", COLORS["purple"]),
    (
        "electricity_trading_benchmark_EUR_per_MWh_el",
        "Gas-based electricity benchmark",
        COLORS["dark"],
    ),
]:
    if col in dispatch.columns:
        fig.add_trace(
            go.Scatter(
                x=dispatch["datetime"],
                y=series(dispatch, col),
                mode="lines",
                name=label,
                line=dict(color=color, width=1.5),
            )
        )
fig.update_yaxes(title_text="EUR/MWh_el")
fig.update_xaxes(title_text="Time")
finish(fig, "Energy price composition and gas benchmark", 560).show()

stats = []
for col in [
    "day_ahead_price_EUR_per_MWh",
    "day_ahead_delivered_price_EUR_per_MWh",
    "IDC_delivered_price_EUR_per_MWh",
    "afrr_energy_delivered_price_EUR_per_MWh",
    "electricity_trading_benchmark_EUR_per_MWh_el",
]:
    if col in dispatch.columns:
        values = series(dispatch, col).replace([np.inf, -np.inf], np.nan).dropna()
        if not values.empty:
            stats.append(
                {
                    "series": col.replace("_", " "),
                    "mean": values.mean(),
                    "min": values.min(),
                    "max": values.max(),
                    "p10": values.quantile(0.1),
                    "p90": values.quantile(0.9),
                }
            )
style_table(pd.DataFrame(stats), "Price statistics [EUR/MWh_el]", precision=2)

if "day_ahead_price_EUR_per_MWh" in dispatch.columns:
    monthly_da = month_mean(dispatch, "day_ahead_price_EUR_per_MWh")
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.18,
        subplot_titles=("Time-series", "Monthly average"),
    )
    fig.add_trace(
        go.Scatter(
            x=dispatch["datetime"],
            y=series(dispatch, "day_ahead_price_EUR_per_MWh"),
            mode="lines",
            name="DA price",
            line=dict(color=COLORS["blue"], width=1.2),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=monthly_da.index,
            y=monthly_da.values,
            name="Monthly average",
            marker_color=COLORS["teal"],
        ),
        row=2,
        col=1,
    )
    fig.update_yaxes(title_text="EUR/MWh", row=1, col=1)
    fig.update_yaxes(title_text="EUR/MWh", row=2, col=1)
    finish(fig, "Day-ahead electricity price profile", 720).show()

if not plants.empty:
    keep = [
        c
        for c in [
            "name",
            "unit_type",
            "technology",
            "fuel_type",
            "max_power",
            "efficiency",
            "max_capacity",
            "max_power_charge",
            "max_power_discharge",
            "initial_soc",
            "efficiency_charge",
            "efficiency_discharge",
        ]
        if c in plants.columns
    ]
    style_table(plants[keep], "Plant technology configuration", precision=3)
if not additional_charges.empty:
    style_table(additional_charges, "Additional electricity charges [EUR/MWh_el]", precision=2)

# 3. Operations

Demand coverage, sequential market positions, ETES storage operation, and aFRR capacity behaviour.


In [ ]:
coverage = pd.DataFrame(
    [
        {"source": "Gas boiler", "heat_MWh": report_row["gas_heat_MWh"]},
        {"source": "ETES / storage discharge", "heat_MWh": report_row["electric_heat_MWh"]},
    ]
)
fig = go.Figure(
    go.Bar(
        x=coverage["source"],
        y=coverage["heat_MWh"],
        marker_color=[COLORS["red"], COLORS["green"]],
        text=coverage["heat_MWh"].round(0),
        texttemplate="%{text:,.0f} MWh",
        textposition="outside",
    )
)
fig.update_yaxes(title_text="Heat supplied [MWh_th]")
finish(fig, "Annual heat demand coverage by supply type", 520).show()

monthly_heat = pd.DataFrame(
    {
        "Gas boiler": month_sum(dispatch, "gas_heat_MWh"),
        "ETES / storage discharge": month_sum(dispatch, "etes_discharge_MWh"),
    }
).fillna(0.0)
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=monthly_heat.index,
        y=monthly_heat["Gas boiler"],
        name="Gas boiler",
        marker_color=COLORS["red"],
    )
)
fig.add_trace(
    go.Bar(
        x=monthly_heat.index,
        y=monthly_heat["ETES / storage discharge"],
        name="ETES / storage discharge",
        marker_color=COLORS["green"],
    )
)
fig.update_layout(barmode="stack")
fig.update_yaxes(title_text="Monthly heat [MWh_th]")
fig.update_xaxes(title_text="Month")
finish(fig, "Monthly heat demand coverage", 560).show()

## Electricity Balance

In [ ]:
da_mwh = float(series(dispatch, "DA_position_MWh").sum())
idc_buy = float(series(dispatch, "IDC_buy_MWh").sum())
idc_sell = float(series(dispatch, "IDC_sell_MWh").sum())
idc_net = idc_buy - idc_sell
da_mwh_net = da_mwh - idc_sell  # DA procurement net of IDC sell reduction
afrr_mwh = float(series(dispatch, "afrr_energy_activated_MWh").sum())
total_mwh = float(series(dispatch, "electricity_consumption_MWh").sum())

# Balance table
elec_balance = pd.DataFrame(
    [
        {
            "Market channel": "Day-Ahead procurement",
            "Volume (MWh_el)": da_mwh,
            "Share (%)": 100 * da_mwh / total_mwh,
        },
        {"Market channel": "DA net", "Volume (MWh_el)": da_mwh_net, "Share (%)": 100.0},
        {
            "Market channel": "IDC buy",
            "Volume (MWh_el)": idc_buy,
            "Share (%)": 100 * idc_buy / total_mwh,
        },
        {
            "Market channel": "IDC sell (reduction)",
            "Volume (MWh_el)": -idc_sell,
            "Share (%)": -100 * idc_sell / total_mwh,
        },
        {
            "Market channel": "IDC net",
            "Volume (MWh_el)": idc_net,
            "Share (%)": 100 * idc_net / total_mwh,
        },
        {
            "Market channel": "aFRR energy activated",
            "Volume (MWh_el)": afrr_mwh,
            "Share (%)": 100 * afrr_mwh / total_mwh,
        },
        {"Market channel": "Total consumption", "Volume (MWh_el)": total_mwh, "Share (%)": 100.0},
    ]
)
style_table(
    elec_balance.set_index("Market channel"),
    "Electricity balance by market channel [MWh_el]",
    precision=1,
)

# Annual stacked bar
fig = go.Figure()
for val, label, color in [
    (da_mwh_net, "Day-Ahead net", COLORS["teal"]),
    (idc_buy, "IDC buy", COLORS["orange"]),
    (afrr_mwh, "aFRR energy", COLORS["green"]),
]:
    fig.add_trace(
        go.Bar(
            x=["Electricity Procurement"],
            y=[val],
            name=label,
            marker_color=color,
            text=[f"{val:,.0f} MWh<br>({100 * val / total_mwh:.1f}%)"],
            textposition="inside",
            textfont=dict(color="white", size=13),
            hovertemplate=f"<b>{label}</b><br>%{{y:,.0f}} MWh_el<extra></extra>",
        )
    )
fig.update_layout(
    barmode="stack",
    height=300,
    margin=dict(l=160, r=45, t=60, b=45),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
    yaxis_title="MWh_el",
)
finish(fig, "Annual Electricity Procurement by Market Channel", 300).show()

# Monthly breakdown by market
monthly_da = month_sum(dispatch, "DA_position_MWh")
monthly_idc_buy = month_sum(dispatch, "IDC_buy_MWh")
monthly_idc_sell = month_sum(dispatch, "IDC_sell_MWh")
monthly_da_net = monthly_da - monthly_idc_sell
monthly_afrr = month_sum(dispatch, "afrr_energy_activated_MWh")

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_names = [months[i] if i < len(months) else "" for i in range(len(monthly_da))]

fig = go.Figure()
for y_vals, label, color in [
    (monthly_da_net, "Day-Ahead net", COLORS["teal"]),
    (monthly_idc_buy, "IDC buy", COLORS["orange"]),
    (monthly_afrr, "aFRR energy", COLORS["green"]),
]:
    fig.add_trace(
        go.Bar(
            x=month_names,
            y=y_vals,
            name=label,
            marker_color=color,
            hovertemplate=f"<b>{label}</b><br>%{{y:,.0f}} MWh_el<extra></extra>",
        )
    )
fig.update_layout(barmode="stack")
fig.update_yaxes(title_text="MWh_el")
fig.update_xaxes(title_text="Month")
finish(fig, "Monthly Electricity Procurement by Market Channel", 520).show()

## 3.1 Demand Coverage

In [ ]:
# Split electric heat by simultaneous charge+discharge (direct pass-through) vs pure storage use
_charge = series(dispatch, "etes_charge_MWh")
_discharge = series(dispatch, "etes_discharge_MWh")
_gas = series(dispatch, "gas_heat_MWh")

_mask_direct = (_charge > 0) & (_discharge > 0)
_mask_storage = (_charge == 0) & (_discharge > 0)

dispatch["_direct_heat_MWh"] = _discharge.where(_mask_direct, 0.0)
dispatch["_storage_heat_MWh"] = _discharge.where(_mask_storage, 0.0)

direct_heat_annual = float(dispatch["_direct_heat_MWh"].sum())
storage_heat_annual = float(dispatch["_storage_heat_MWh"].sum())
gas_heat_annual = float(_gas.sum())
total_heat_annual = direct_heat_annual + storage_heat_annual + gas_heat_annual

direct_pct = 100.0 * direct_heat_annual / total_heat_annual if total_heat_annual > 0 else 0.0
storage_pct = 100.0 * storage_heat_annual / total_heat_annual if total_heat_annual > 0 else 0.0
gas_pct = 100.0 * gas_heat_annual / total_heat_annual if total_heat_annual > 0 else 0.0

# 2.1 – horizontal stacked bar (three segments)
fig = go.Figure()
for val, pct, color, name, font_color in [
    (direct_heat_annual, direct_pct, COLORS["dark"], "Direct Supply by EH", "white"),
    (storage_heat_annual, storage_pct, COLORS["green"], "Storage Discharge", "white"),
    (gas_heat_annual, gas_pct, "#B0BEC5", "Gas Boiler", "#263238"),
]:
    fig.add_trace(
        go.Bar(
            x=[val],
            y=["Thermal Demand Coverage"],
            orientation="h",
            marker_color=color,
            name=name,
            text=[f"{val:,.0f} MWh<br>({pct:.1f}%)"],
            textposition="inside",
            textfont=dict(color=font_color, size=13),
            hovertemplate=f"<b>{name}</b><br>%{{x:,.0f}} MWh<extra></extra>",
        )
    )
fig.update_layout(
    barmode="stack",
    xaxis_title="Energy (MWh)",
    yaxis_title="",
    margin=dict(l=160, r=45, t=60, b=45),
    height=280,
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
)
finish(fig, "2.1 Annual Demand Coverage by Supply Type", 280).show()


# 2.2 – monthly stacked bars with electrification rate trend
def _month_sum_dispatch(col: str) -> pd.Series:
    tmp = pd.DataFrame({"datetime": dispatch["datetime"], "val": dispatch[col]})
    return tmp.dropna(subset=["datetime"]).set_index("datetime")["val"].resample("ME").sum()


monthly_direct = _month_sum_dispatch("_direct_heat_MWh")
monthly_storage = _month_sum_dispatch("_storage_heat_MWh")
monthly_gas = _month_sum_dispatch("gas_heat_MWh")
monthly_total = monthly_direct + monthly_storage + monthly_gas
monthly_electrified = ((monthly_direct + monthly_storage) / monthly_total * 100.0).fillna(0.0)

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_names = [months[i] if i < len(months) else "" for i in range(len(monthly_gas))]

fig = make_subplots(specs=[[{"secondary_y": True}]])
for y_vals, color, name in [
    (monthly_direct, COLORS["dark"], "Direct Supply by EH"),
    (monthly_storage, COLORS["green"], "Storage Discharge"),
    (monthly_gas, "#B0BEC5", "Gas Boiler"),
]:
    fig.add_trace(
        go.Bar(
            x=month_names,
            y=y_vals,
            name=name,
            marker_color=color,
            hovertemplate=f"<b>{name}</b><br>%{{y:,.0f}} MWh<extra></extra>",
        ),
        secondary_y=False,
    )
fig.add_trace(
    go.Scatter(
        x=month_names,
        y=monthly_electrified,
        name="Electrified Heat Supply",
        mode="lines+markers",
        line=dict(color="#111", width=3),
        marker=dict(size=8),
        hovertemplate="<b>Electrified Share</b><br>%{y:.1f}%<extra></extra>",
    ),
    secondary_y=True,
)
fig.update_layout(barmode="stack", height=560)
fig.update_yaxes(title_text="Energy (MWh)", secondary_y=False)
fig.update_yaxes(title_text="Electrified Share (%)", secondary_y=True, range=[0, 100])
fig.update_xaxes(title_text="Month")
finish(fig, "2.2 Monthly Demand Coverage", 560).show()

In [ ]:
PROFILE_WEEK_START = None  # Example: "2025-08-25". Keep None for automatic active-week selection.


def select_active_week(
    frame: pd.DataFrame, manual: str | None = None
) -> tuple[pd.Timestamp, pd.Timestamp]:
    if manual:
        start = pd.Timestamp(manual)
        return start, start + pd.Timedelta(days=7)
    activity_cols = [
        "day_ahead_position_MWh_el",
        "intraday_buy_MWh_el",
        "intraday_sell_MWh_el",
        "afrr_energy_activated_MWh_el",
        "afrr_capacity_reserved_MWh",
    ]
    data = frame.copy()
    data["activity"] = sum(series(data, c).abs() for c in activity_cols)
    weekly = data.groupby(pd.Grouper(key="datetime", freq="W-MON"))["activity"].sum()
    start = (
        (weekly.idxmax() - pd.Timedelta(days=6)).normalize()
        if not weekly.empty
        else data["datetime"].min().normalize()
    )
    return start, start + pd.Timedelta(days=7)


week_start, week_end = select_active_week(market, PROFILE_WEEK_START)
week = market[(market["datetime"] >= week_start) & (market["datetime"] < week_end)].copy()
week_dispatch = dispatch[
    (dispatch["datetime"] >= week_start) & (dispatch["datetime"] < week_end)
].copy()

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Day-ahead baseline",
        "Intraday adjustment",
        "aFRR energy and final electricity",
        "Gas boiler and ETES SoC",
    ),
    specs=[[{}], [{}], [{}], [{"secondary_y": True}]],
)
x = week["datetime"]
fig.add_trace(
    go.Bar(
        x=x,
        y=series(week, "day_ahead_position_MWh_el"),
        name="DA position",
        marker_color="rgba(76,120,168,0.45)",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=series(week, "day_ahead_position_MWh_el"),
        mode="lines",
        name="DA line",
        line=dict(color=COLORS["blue"], width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=x,
        y=series(week, "intraday_buy_MWh_el"),
        name="IDC buy",
        marker_color="rgba(0,218,157,0.60)",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=x,
        y=-series(week, "intraday_sell_MWh_el"),
        name="IDC sell/reduction",
        marker_color="rgba(243,52,1,0.55)",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=series(week, "scheduled_electricity_procurement_MWh_el"),
        mode="lines",
        name="Scheduled DA+IDC",
        line=dict(color="#111", width=2),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=x,
        y=series(week, "afrr_energy_activated_MWh_el"),
        name="aFRR activated",
        marker_color="rgba(139,111,179,0.60)",
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=x,
        y=series(week, "afrr_capacity_reserved_MWh"),
        name="Reserved headroom",
        marker_color="rgba(157,127,100,0.35)",
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=series(week, "actual_electricity_consumption_MWh_el"),
        mode="lines",
        name="Actual electricity",
        line=dict(color="#111", width=2, dash="dash"),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=x,
        y=series(week, "gas_heat_output_MWh_th"),
        name="Gas boiler heat",
        marker_color="rgba(243,52,1,0.45)",
    ),
    row=4,
    col=1,
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=week_dispatch["datetime"],
        y=series(week_dispatch, "etes_soc_MWh"),
        mode="lines",
        name="ETES SoC",
        line=dict(color=COLORS["dark"], width=2),
    ),
    row=4,
    col=1,
    secondary_y=True,
)
fig.update_layout(barmode="relative")
fig.update_yaxes(title_text="MWh_el", row=1, col=1)
fig.update_yaxes(title_text="MWh_el", row=2, col=1)
fig.update_yaxes(title_text="MWh_el", row=3, col=1)
fig.update_yaxes(title_text="MWh_th", row=4, col=1, secondary_y=False)
fig.update_yaxes(title_text="SoC [MWh_th]", row=4, col=1, secondary_y=True)
finish(
    fig,
    f"Sequential electricity and plant operation profile: {week_start.date()} to {(week_end - pd.Timedelta(days=1)).date()}",
    980,
).show()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=("ETES charging and discharging", "Storage state of charge"),
)
fig.add_trace(
    go.Bar(
        x=dispatch["datetime"],
        y=series(dispatch, "etes_charge_MWh"),
        name="ETES charge",
        marker_color="rgba(0,218,157,0.55)",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=dispatch["datetime"],
        y=-series(dispatch, "etes_discharge_MWh"),
        name="ETES discharge",
        marker_color="rgba(242,142,43,0.55)",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=dispatch["datetime"],
        y=series(dispatch, "etes_soc_MWh"),
        mode="lines",
        fill="tozeroy",
        name="ETES SoC",
        line=dict(color=COLORS["dark"], width=2),
        fillcolor="rgba(14,62,62,0.18)",
    ),
    row=2,
    col=1,
)
fig.update_yaxes(title_text="Storage flow [MWh]", row=1, col=1)
fig.update_yaxes(title_text="SoC [MWh_th]", row=2, col=1)
finish(fig, "ETES storage operation", 720).show()

if not capacity.empty:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=capacity["block_start"],
            y=series(capacity, "capacity_price_EUR_per_MW_h"),
            mode="lines+markers",
            name="Capacity price",
            line=dict(color=COLORS["dark"], width=2),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Bar(
            x=capacity["block_start"],
            y=series(capacity, "reserved_capacity_MW"),
            name="Reserved capacity",
            marker_color="rgba(157,127,100,0.55)",
        ),
        secondary_y=True,
    )
    fig.add_trace(
        go.Bar(
            x=capacity["block_start"],
            y=series(capacity, "total_afrr_energy_activated_MWh_in_block"),
            name="Activated energy in block",
            marker_color="rgba(139,111,179,0.45)",
        ),
        secondary_y=True,
    )
    fig.update_yaxes(title_text="EUR/MW/h", secondary_y=False)
    fig.update_yaxes(title_text="MW or MWh/block", secondary_y=True)
    finish(fig, "aFRR down capacity market: price, reservation, and activation", 620).show()
else:
    display(HTML("<b>aFRR capacity summary is not available for this report scenario.</b>"))

# 4. Financials

Cashflow waterfall, monthly market value allocation, scenario cost comparison, and market volumes.


In [ ]:
benchmark = report_row["gas_only_benchmark_cost_EUR"]
fig = go.Figure(
    go.Waterfall(
        measure=[
            "absolute",
            "relative",
            "relative",
            "relative",
            "relative",
            "relative",
            "relative",
            "relative",
            "total",
        ],
        x=[
            "Gas-only\nbenchmark",
            "Gas\ndisplacement",
            "DA\nmarket",
            "IDC\nbuy",
            "IDC\nsell",
            "aFRR\nenergy",
            "aFRR\ncapacity",
            "Electricity\ncharges",
            "Net\ncost",
        ],
        y=[
            kEUR(benchmark),
            kEUR(report_row["gas_cost_EUR"] - benchmark),
            kEUR(report_row["DA_market_cost_EUR"]),
            kEUR(report_row["IDC_buy_cost_EUR"]),
            -kEUR(report_row["IDC_sell_revenue_EUR"]),
            kEUR(report_row["afrr_energy_cost_EUR"]),
            -kEUR(report_row["afrr_capacity_revenue_EUR"]),
            kEUR(report_row["additional_electricity_charges_EUR"]),
            kEUR(report_row["net_operating_cost_EUR"]),
        ],
        increasing={"marker": {"color": COLORS["red"]}},
        decreasing={"marker": {"color": COLORS["green"]}},
        totals={"marker": {"color": COLORS["dark"]}},
        connector={"line": {"color": "#90A4AE"}},
        texttemplate="%{y:,.0f}",
        textposition="outside",
    )
)
fig.update_yaxes(title_text="Cost / revenue [kEUR]")
finish(fig, "Cashflow waterfall from gas-only benchmark to net operating cost", 620).show()

m = dispatch.copy()
if "electricity_trading_benchmark_EUR_per_MWh_el" in m.columns:
    bench = series(m, "electricity_trading_benchmark_EUR_per_MWh_el")
else:
    bench = pd.Series(0.0, index=m.index)
m["DA value"] = series(m, "DA_position_MWh") * (
    bench - series(m, "day_ahead_delivered_price_EUR_per_MWh")
).clip(lower=0)
m["IDC value"] = series(m, "IDC_buy_MWh") * (
    bench - series(m, "IDC_delivered_price_EUR_per_MWh")
).clip(lower=0) + series(m, "IDC_sell_revenue_EUR")
m["aFRR Capacity"] = series(m, "afrr_capacity_revenue_EUR")
m["aFRR Energy"] = series(m, "afrr_energy_net_value_after_charges_EUR")
monthly_value = m.groupby(pd.Grouper(key="datetime", freq="ME"))[
    ["DA value", "IDC value", "aFRR Capacity", "aFRR Energy"]
].sum()
fig = go.Figure()
for col, color in [
    ("DA value", COLORS["blue"]),
    ("IDC value", COLORS["orange"]),
    ("aFRR Capacity", COLORS["dark"]),
    ("aFRR Energy", COLORS["green"]),
]:
    fig.add_trace(
        go.Bar(x=monthly_value.index, y=monthly_value[col].map(kEUR), name=col, marker_color=color)
    )
fig.update_layout(barmode="relative")
fig.update_yaxes(title_text="Monthly value [kEUR]")
finish(fig, "Monthly market value allocation", 620).show()

In [ ]:
m = dispatch.copy()
bench = series(m, "electricity_trading_benchmark_EUR_per_MWh_el")

# Savings vs gas baseline per market channel
m["DA savings"] = series(m, "DA_position_MWh") * (
    bench - series(m, "day_ahead_delivered_price_EUR_per_MWh")
).clip(lower=0)

m["IDC savings"] = series(m, "IDC_buy_MWh") * (
    bench - series(m, "IDC_delivered_price_EUR_per_MWh")
).clip(lower=0) + series(m, "IDC_sell_MWh") * (
    series(m, "IDC_delivered_price_EUR_per_MWh") - bench
).clip(lower=0)

m["aFRR Capacity savings"] = series(m, "afrr_capacity_revenue_EUR")
m["aFRR Energy savings"] = series(m, "afrr_energy_net_value_after_charges_EUR").clip(lower=0)

monthly_savings = m.groupby(pd.Grouper(key="datetime", freq="ME"))[
    ["DA savings", "IDC savings", "aFRR Capacity savings", "aFRR Energy savings"]
].sum()

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_names = [months[i] if i < len(months) else "" for i in range(len(monthly_savings))]

annual_total = monthly_savings.sum().sum()

fig = go.Figure()
for col, label, color in [
    ("DA savings", "Day-Ahead", COLORS["teal"]),
    ("IDC savings", "Intraday Continuous", COLORS["orange"]),
    ("aFRR Capacity savings", "aFRR Capacity", COLORS["dark"]),
    ("aFRR Energy savings", "aFRR Energy", COLORS["green"]),
]:
    fig.add_trace(
        go.Bar(
            x=month_names,
            y=monthly_savings[col],
            name=label,
            marker_color=color,
            hovertemplate=f"<b>{label}</b><br>%{{y:,.0f}} €<extra></extra>",
        )
    )

fig.add_annotation(
    x=1.0,
    y=1.02,
    xref="paper",
    yref="paper",
    text=f"<b>Annual total: €{annual_total:,.0f}</b>",
    showarrow=False,
    font=dict(size=13, color=COLORS["dark"]),
    xanchor="right",
)

fig.update_layout(
    barmode="stack",
    legend=dict(orientation="v", yanchor="top", y=0.99, xanchor="left", x=0.01),
)
fig.update_yaxes(title_text="Savings vs Gas (€)")
fig.update_xaxes(title_text="Month")
finish(fig, "3.2 Monthly Market Savings Allocation by Market", 560).show()

In [ ]:
plot_df = analysis_df.copy()
fig = go.Figure()
for col, label, color in [
    ("gas_cost_EUR", "Gas cost", COLORS["red"]),
    ("DA_market_cost_EUR", "DA market cost", COLORS["blue"]),
    ("IDC_buy_cost_EUR", "IDC buy cost", COLORS["green"]),
    ("afrr_energy_cost_EUR", "aFRR energy settlement", COLORS["purple"]),
    ("additional_electricity_charges_EUR", "Electricity charges", COLORS["orange"]),
    ("IDC_sell_revenue_EUR", "IDC sell revenue", COLORS["cyan"]),
    ("afrr_capacity_revenue_EUR", "aFRR capacity revenue", COLORS["brown"]),
]:
    values = plot_df[col].copy()
    if "revenue" in col:
        values = -values
    fig.add_trace(go.Bar(x=plot_df["scenario"], y=values.map(kEUR), name=label, marker_color=color))
fig.add_trace(
    go.Scatter(
        x=plot_df["scenario"],
        y=plot_df["net_operating_cost_EUR"].map(kEUR),
        mode="lines+markers+text",
        name="Net operating cost",
        line=dict(color="#111", width=3),
        marker=dict(size=10),
        text=plot_df["net_operating_cost_EUR"].map(lambda x: f"{kEUR(x):,.0f}"),
        textposition="top center",
    )
)
fig.update_layout(barmode="relative")
fig.update_yaxes(title_text="Cost / revenue [kEUR]")
finish(fig, "Scenario comparison: costs, revenues, and net operating cost", 720).show()

fig = make_subplots(specs=[[{"secondary_y": True}]])
for col, label, color in [
    ("DA_procured_MWh_el", "DA procured", COLORS["blue"]),
    ("IDC_sell_MWh_el", "DA sold/reduced in IDC", COLORS["cyan"]),
    ("IDC_buy_MWh_el", "IDC procured", COLORS["green"]),
    ("afrr_energy_activated_MWh_el", "aFRR energy activated", COLORS["purple"]),
]:
    fig.add_trace(
        go.Bar(x=plot_df["scenario"], y=plot_df[col], name=label, marker_color=color),
        secondary_y=False,
    )
fig.add_trace(
    go.Bar(
        x=plot_df["scenario"],
        y=plot_df["afrr_capacity_reserved_MW_h"],
        name="aFRR capacity reserved",
        marker=dict(
            color="rgba(255,255,255,0.35)",
            line=dict(color="#111", width=2),
            pattern=dict(shape="/"),
        ),
    ),
    secondary_y=True,
)
fig.update_layout(barmode="group")
fig.update_yaxes(title_text="Energy volume [MWh_el]", secondary_y=False)
fig.update_yaxes(title_text="aFRR capacity reservation [MW·h]", secondary_y=True)
finish(fig, "Trade activity volumes by scenario", 680).show()

fig = go.Figure(
    go.Bar(
        x=plot_df["scenario"],
        y=plot_df["net_operating_cost_EUR"].map(kEUR),
        marker_color=COLORS["teal"],
        text=plot_df["net_operating_cost_EUR"].map(lambda x: f"{kEUR(x):,.0f}"),
        texttemplate="%{text} kEUR",
        textposition="outside",
    )
)
fig.update_yaxes(title_text="Net operating cost [kEUR]")
finish(fig, "Net operating cost by scenario", 560).show()

## 4.1 Grid Fees

German network charges settled ex-post from the realized grid withdrawal: the full-load-hour-tiered grid **energy** charge, the grid **capacity** charge billed on the **high-load-window peak** under §19(2) StromNEV atypical grid use, the **special network use** group A/B split, and flat **levies & taxes**. The assumed vs realized full-load-hour tier. If it did not hold, the bill is corrected ex-post and the run should be repeated with the warned tier.

In [ ]:
# 4.1 Grid Fees (Netzentgelte) - ex-post network-charge settlement
def _sval(frame, col, default="n/a"):
    if col in frame.columns and len(frame):
        return str(frame[col].iloc[0])
    return default


if "grid_fee_total_EUR" not in summary.columns:
    display(
        HTML(
            "<b>Grid-fee settlement is not available for this scenario.</b> "
            "Re-run the case with additional charges enabled to populate grid_fee_* indicators."
        )
    )
else:
    energy_eur = value(summary, "grid_fee_energy_charge_EUR")
    capacity_eur = value(summary, "grid_fee_capacity_charge_EUR")
    special_eur = value(summary, "grid_fee_special_network_use_EUR")
    levies_eur = value(summary, "grid_fee_levies_EUR")
    total_eur = value(summary, "grid_fee_total_EUR")
    annual_peak = value(summary, "grid_annual_peak_MW")
    window_peak = value(summary, "grid_high_load_window_peak_MW")
    billed_peak = value(summary, "grid_billed_peak_MW")
    flh = value(summary, "grid_full_load_hours")
    assumed_tier = _sval(summary, "grid_assumed_tier")
    realized_tier = _sval(summary, "grid_realized_tier")
    held = _sval(summary, "grid_tier_assumption_held", "True") in ("True", "true", "1", "1.0")
    net_incl = value(summary, "net_operating_cost_incl_grid_fees_EUR")

    display(
        cards(
            [
                ("Total grid fee", f"{kEUR(total_eur):,.0f} kEUR", "Netzentgelte + levies + taxes"),
                (
                    "Grid energy charge",
                    f"{kEUR(energy_eur):,.0f} kEUR",
                    f"full-load-hour tier '{realized_tier}'",
                ),
                (
                    "Grid capacity charge",
                    f"{kEUR(capacity_eur):,.0f} kEUR",
                    f"billed peak {billed_peak:,.2f} MW",
                ),
                (
                    "Special network use",
                    f"{kEUR(special_eur):,.0f} kEUR",
                    "group A (1st GWh) + group B",
                ),
                (
                    "Levies & taxes",
                    f"{kEUR(levies_eur):,.0f} kEUR",
                    "CHP, offshore, concession, e-tax",
                ),
                (
                    "Net cost incl. grid fees",
                    f"{kEUR(net_incl):,.0f} kEUR",
                    "operating cost + grid fees",
                ),
            ]
        )
    )

    tier_note = f"Full-load hours <b>{flh:,.0f} h/a</b> &rarr; tier <b>'{realized_tier}'</b>. " + (
        "Assumed tier held."
        if held
        else f"<span style='color:{COLORS['red']};'>Assumed tier '{assumed_tier}' did NOT hold; "
        f"bill corrected ex-post &mdash; re-run with --assumed-grid-tier {realized_tier} "
        "for a self-consistent dispatch.</span>"
    )
    atypical = (
        "Atypical grid use (&sect;19(2) StromNEV): capacity billed on the high-load-window peak "
        f"<b>{window_peak:,.2f} MW</b> instead of the annual peak <b>{annual_peak:,.2f} MW</b>."
    )
    display(
        HTML(
            f"<div style='margin:6px 0 16px 0;color:#37474F;font-size:13px;'>{tier_note}<br>{atypical}</div>"
        )
    )

    fig = go.Figure(
        go.Bar(
            x=[kEUR(energy_eur), kEUR(capacity_eur), kEUR(special_eur), kEUR(levies_eur)],
            y=[
                "Grid energy charge",
                "Grid capacity charge",
                "Special network use",
                "Levies & taxes",
            ],
            orientation="h",
            marker_color=[COLORS["blue"], COLORS["dark"], COLORS["orange"], COLORS["green"]],
            text=[
                f"{kEUR(v):,.0f} kEUR" for v in [energy_eur, capacity_eur, special_eur, levies_eur]
            ],
            textposition="auto",
        )
    )
    fig.update_xaxes(title_text="Cost [kEUR]")
    finish(fig, "Annual grid-fee breakdown by component", 360).show()

    fig = go.Figure(
        go.Bar(
            x=["Annual peak", "High-load-window peak", "Billed peak"],
            y=[annual_peak, window_peak, billed_peak],
            marker_color=[COLORS["blue"], COLORS["red"], COLORS["dark"]],
            text=[f"{v:,.2f} MW" for v in [annual_peak, window_peak, billed_peak]],
            textposition="outside",
        )
    )
    fig.update_yaxes(title_text="Power [MW]")
    finish(fig, "Capacity-charge peak basis (atypical grid use)", 420).show()

### 4.2 How grid draw avoids the high-load windows

Average **weekday** load profile by hour of day for the seasons that have a DSO high-load window (Winter and Autumn). The shaded red bands are the high-load windows: grid draw (e-heater charging) collapses to **zero** inside them, while the gas boiler and pre-charged storage carry the heat — the behaviour that drives the billed capacity peak, and therefore the §19(2) capacity charge, to zero.

In [ ]:
# 4.2 Atypical grid use - average weekday profile around the high-load windows
CASE_TZ = "Europe/Berlin"  # German case; window rules are defined in local wall-clock


def _high_load_window_flag(index):
    """0/1 German high-load-window flag for a local-wall-clock DatetimeIndex."""
    idx = pd.DatetimeIndex(index)
    mins = idx.hour.to_numpy() * 60 + idx.minute.to_numpy()
    mon = idx.month.to_numpy()
    weekday = idx.weekday.to_numpy() < 5
    winter = np.isin(mon, [1, 2, 12]) & (mins >= 9 * 60 + 45) & (mins < 19 * 60 + 30)
    autumn = np.isin(mon, [9, 10, 11]) & (
        ((mins >= 11 * 60 + 15) & (mins < 13 * 60 + 45))
        | ((mins >= 15 * 60) & (mins < 19 * 60 + 15))
    )
    return (weekday & (winter | autumn)).astype(int)


prof = dispatch[
    ["datetime", "actual_electricity_consumption_MWh", "gas_heat_MWh", "etes_discharge_MWh"]
].copy()
# dispatch datetimes are stored tz-aware and parsed to UTC-naive; recover local wall-clock.
_dt_local = (
    pd.to_datetime(prof["datetime"])
    .dt.tz_localize("UTC")
    .dt.tz_convert(CASE_TZ)
    .dt.tz_localize(None)
)
_step_h = float(
    (_dt_local.sort_values().iloc[1] - _dt_local.sort_values().iloc[0]).total_seconds() / 3600
)
prof["hod"] = _dt_local.dt.hour + _dt_local.dt.minute / 60.0
prof["month"] = _dt_local.dt.month
prof["is_weekday"] = _dt_local.dt.weekday < 5
prof["high_load_window"] = _high_load_window_flag(pd.DatetimeIndex(_dt_local))
prof["grid_draw_MW"] = prof["actual_electricity_consumption_MWh"] / _step_h
prof["gas_MW"] = prof["gas_heat_MWh"] / _step_h
prof["disch_MW"] = prof["etes_discharge_MWh"] / _step_h

seasons = [
    ("Winter weekdays (Jan / Feb / Dec)", [1, 2, 12], [(9.75, 19.5)]),
    ("Autumn weekdays (Sep / Oct / Nov)", [9, 10, 11], [(11.25, 13.75), (15.0, 19.25)]),
]
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.13, subplot_titles=[s[0] for s in seasons]
)
for i, (_label, months, bands) in enumerate(seasons, start=1):
    sub = prof[prof["is_weekday"] & prof["month"].isin(months)]
    if sub.empty:
        continue
    grouped = sub.groupby("hod")
    draw = grouped["grid_draw_MW"].mean()
    gas = grouped["gas_MW"].mean()
    disch = grouped["disch_MW"].mean()
    for x0, x1 in bands:
        fig.add_vrect(x0=x0, x1=x1, fillcolor="rgba(243,52,1,0.10)", line_width=0, row=i, col=1)
    fig.add_trace(
        go.Scatter(
            x=draw.index,
            y=draw.values,
            mode="lines",
            name="Grid draw (e-heater)",
            line=dict(color=COLORS["teal"], width=2.5),
            fill="tozeroy",
            fillcolor="rgba(34,151,151,0.25)",
            legendgroup="draw",
            showlegend=(i == 1),
        ),
        row=i,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=gas.index,
            y=gas.values,
            mode="lines",
            name="Gas boiler heat",
            line=dict(color=COLORS["red"], width=2),
            legendgroup="gas",
            showlegend=(i == 1),
        ),
        row=i,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=disch.index,
            y=disch.values,
            mode="lines",
            name="Storage discharge",
            line=dict(color=COLORS["dark"], width=2, dash="dot"),
            legendgroup="disch",
            showlegend=(i == 1),
        ),
        row=i,
        col=1,
    )
    fig.update_yaxes(title_text="Power [MW]", row=i, col=1)
fig.update_xaxes(
    title_text="Hour of day (local)",
    tickmode="linear",
    tick0=0,
    dtick=2,
    range=[0, 24],
    row=2,
    col=1,
)
finish(fig, "Atypical grid use: average weekday profile (shaded = high-load window)", 720).show()

mask_in = prof["high_load_window"] == 1
energy_in = float(prof.loc[mask_in, "actual_electricity_consumption_MWh"].sum())
energy_total = float(prof["actual_electricity_consumption_MWh"].sum())
peak_in = float(np.nan_to_num(prof.loc[mask_in, "grid_draw_MW"].max()))
peak_all = float(np.nan_to_num(prof["grid_draw_MW"].max()))
share_in = 100.0 * energy_in / energy_total if energy_total else 0.0
display(
    HTML(
        "<div style='margin:6px 0 18px 0;color:#37474F;font-size:13px;'>"
        f"Grid energy drawn inside high-load windows: <b>{energy_in:,.0f} MWh</b> "
        f"(<b>{share_in:.2f}%</b> of {energy_total:,.0f} MWh). "
        f"Peak grid draw in-window <b>{peak_in:,.2f} MW</b> vs annual peak <b>{peak_all:,.2f} MW</b> "
        "&mdash; the billed capacity peak under §19(2) collapses to the in-window peak."
        "</div>"
    )
)

## Per-MW e-heater normalized results

In [ ]:
eheater_mw = SIZING.get("eheater_power_MW", np.nan)
if not (np.isfinite(eheater_mw) and eheater_mw > 0):
    display(
        HTML("<b>Installed e-heater power is unavailable; cannot compute per-MW indicators.</b>")
    )
else:
    market_cost_cols = [
        ("gas_cost_EUR", "Gas cost"),
        ("DA_market_cost_EUR", "DA market cost"),
        ("IDC_buy_cost_EUR", "IDC buy cost"),
        ("afrr_energy_cost_EUR", "aFRR energy settlement"),
        ("additional_electricity_charges_EUR", "Electricity charges"),
        ("IDC_sell_revenue_EUR", "IDC sell revenue"),
        ("afrr_capacity_revenue_EUR", "aFRR capacity revenue"),
        ("net_operating_cost_EUR", "Net operating cost"),
        ("cost_savings_vs_gas_only_EUR", "Savings vs gas-only"),
    ]
    per_mw = pd.DataFrame({"scenario": analysis_df["scenario"].astype(str)})
    for col, label in market_cost_cols:
        per_mw[label] = analysis_df[col].to_numpy() / eheater_mw
    style_table(
        per_mw.set_index("scenario"),
        f"Per-market and net results per installed MW e-heater [EUR/MW_el] "
        f"(e-heater = {eheater_mw:g} MW_el)",
        precision=1,
    )

    fig = go.Figure()
    for col, label, color in [
        ("gas_cost_EUR", "Gas cost", COLORS["red"]),
        ("DA_market_cost_EUR", "DA market cost", COLORS["blue"]),
        ("IDC_buy_cost_EUR", "IDC buy cost", COLORS["green"]),
        ("afrr_energy_cost_EUR", "aFRR energy settlement", COLORS["purple"]),
        ("additional_electricity_charges_EUR", "Electricity charges", COLORS["orange"]),
        ("IDC_sell_revenue_EUR", "IDC sell revenue", COLORS["cyan"]),
        ("afrr_capacity_revenue_EUR", "aFRR capacity revenue", COLORS["brown"]),
    ]:
        vals = analysis_df[col].to_numpy() / eheater_mw
        if "revenue" in col:
            vals = -vals
        fig.add_trace(go.Bar(x=analysis_df["scenario"], y=vals, name=label, marker_color=color))
    net_per_mw = analysis_df["net_operating_cost_EUR"].to_numpy() / eheater_mw
    fig.add_trace(
        go.Scatter(
            x=analysis_df["scenario"],
            y=net_per_mw,
            mode="lines+markers+text",
            name="Net operating cost",
            line=dict(color="#111", width=3),
            marker=dict(size=10),
            text=[f"{v:,.0f}" for v in net_per_mw],
            textposition="top center",
        )
    )
    fig.update_layout(barmode="relative")
    fig.update_yaxes(title_text="EUR per installed MW e-heater [EUR/MW_el]")
    finish(fig, "Per-market and net results per installed MW e-heater power", 720).show()

In [ ]:
import os
import subprocess
import sys

if EXPORT_HTML:
    if os.environ.get("FLEXIMOD_REPORT_EXPORT_IN_PROGRESS") == "1":
        pass
    else:
        REPORT_HTML_PATH.parent.mkdir(parents=True, exist_ok=True)
        env = os.environ.copy()
        env["FLEXIMOD_REPORT_EXPORT_IN_PROGRESS"] = "1"
        command = [
            sys.executable,
            "-m",
            "nbconvert",
            "--to",
            "html",
            "--execute",
            "--no-input",
            str(NOTEBOOK_PATH),
            "--output-dir",
            str(REPORT_HTML_PATH.parent),
            "--output",
            REPORT_HTML_PATH.stem,
        ]
        print(f"Exporting HTML report to: {REPORT_HTML_PATH}")
        subprocess.run(command, check=True, cwd=REPO_ROOT, env=env)
        print(f"HTML report exported: {REPORT_HTML_PATH}")
else:
    print("EXPORT_HTML is False. Set it to True to export the standalone HTML report.")